In [2]:
import pandas as pd
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
NVIDIA GeForce RTX 4060 Ti


In [3]:
dataset = pd.read_csv("insurance_pre.csv")
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [4]:
dataset = pd.get_dummies(dataset,dtype=int,drop_first=True)
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [5]:
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='object')

In [6]:
independent = dataset[['age', 'bmi', 'children', 'sex_male', 'smoker_yes']]
independent

,age,bmi,children,sex_male,smoker_yes
0,19,27.900,0,0,1
1,18,33.770,1,1,0
2,28,33.000,3,1,0
3,33,22.705,0,1,0
4,32,28.880,0,1,0
...,...,...,...,...,...
1333,50,30.970,3,1,0
1334,18,31.920,0,0,0
1335,18,36.850,0,0,0
1336,21,25.800,0,0,0


In [7]:
dependent = dataset[["charges"]]
dependent

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520
...,...
1333,10600.54830
1334,2205.98080
1335,1629.83350
1336,2007.94500


In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(independent,dependent, test_size=0.30, random_state=0)

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
param_grid = {
    'criterion':['squared_error','friedman_mse',
              'absolute_error','poisson'
             ],
    'splitter':['best','random' ]
}
grid = GridSearchCV(DecisionTreeRegressor(),param_grid, refit=True,verbose=3, n_jobs=1)


grid.fit(X_train,y_train,)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV 1/5] END criterion=squared_error, splitter=best;, score=0.751 total time=   0.0s
[CV 2/5] END criterion=squared_error, splitter=best;, score=0.548 total time=   0.0s
[CV 3/5] END criterion=squared_error, splitter=best;, score=0.772 total time=   0.0s
[CV 4/5] END criterion=squared_error, splitter=best;, score=0.638 total time=   0.0s
[CV 5/5] END criterion=squared_error, splitter=best;, score=0.660 total time=   0.0s
[CV 1/5] END criterion=squared_error, splitter=random;, score=0.626 total time=   0.0s
[CV 2/5] END criterion=squared_error, splitter=random;, score=0.600 total time=   0.0s
[CV 3/5] END criterion=squared_error, splitter=random;, score=0.680 total time=   0.0s
[CV 4/5] END criterion=squared_error, splitter=random;, score=0.648 total time=   0.0s
[CV 5/5] END criterion=squared_error, splitter=random;, score=0.638 total time=   0.0s
[CV 1/5] END criterion=friedman_mse, splitter=best;, score=0.732 total time=   0

,estimator,DecisionTreeRegressor()
,param_grid,"{'criterion': ['squared_error', 'friedman_mse', ...], 'splitter': ['best', 'random']}"
,scoring,None
,n_jobs,1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'squared_error'


In [10]:
re = grid.cv_results_
print("The R_Score value for best parameter {}:".format(grid.best_params_))

The R_Score value for best parameter {'criterion': 'squared_error', 'splitter': 'best'}:


In [11]:
table = pd.DataFrame.from_dict(re)



In [12]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001801,4.000669e-04,0.000800,3.999710e-04,squared_error,best,"{'criterion': 'squared_error', 'splitter': 'be...",0.750787,0.548045,0.771615,0.638092,0.659566,0.673621,0.080977,1
1,0.001500,4.473421e-04,0.000800,3.999949e-04,squared_error,random,"{'criterion': 'squared_error', 'splitter': 'ra...",0.625821,0.600292,0.680237,0.647812,0.637752,0.638383,0.026267,7
2,0.002001,8.176054e-07,0.001000,3.371748e-07,friedman_mse,best,"{'criterion': 'friedman_mse', 'splitter': 'best'}",0.732492,0.562316,0.773751,0.617644,0.661361,0.669513,0.076263,2
3,0.001400,4.898625e-04,0.000800,4.000187e-04,friedman_mse,random,"{'criterion': 'friedman_mse', 'splitter': 'ran...",0.669556,0.669287,0.664793,0.620826,0.587740,0.642441,0.032909,5
4,0.009200,3.999710e-04,0.000600,4.899403e-04,absolute_error,best,"{'criterion': 'absolute_error', 'splitter': 'b...",0.752107,0.637052,0.606960,0.543368,0.666214,0.641140,0.068828,6
5,0.006801,7.486724e-04,0.000901,4.906718e-04,absolute_error,random,"{'criterion': 'absolute_error', 'splitter': 'r...",0.669205,0.668433,0.691992,0.665856,0.586943,0.656486,0.036026,3
6,0.001999,4.156970e-07,0.001001,5.722046e-07,poisson,best,"{'criterion': 'poisson', 'splitter': 'best'}",0.697605,0.573843,0.725479,0.550817,0.615630,0.632675,0.068236,8
7,0.001400,4.900767e-04,0.000599,4.894733e-04,poisson,random,"{'criterion': 'poisson', 'splitter': 'random'}",0.660754,0.673849,0.626954,0.689705,0.609622,0.652177,0.029660,4


In [13]:
y_pred = grid.predict(X_test)

In [14]:
y_pred

array([10085.846   ,  8930.93455 , 44202.6536  , 13143.86485 ,
        9264.797   ,  7228.21565 ,  1615.7667  , 10381.4787  ,
        7954.517   ,  5253.524   ,  4766.022   , 30284.64294 ,
        7633.7206  ,  4992.3764  , 18246.4955  , 11015.1747  ,
       12124.9924  ,  3292.52985 ,  6455.86265 , 33307.5508  ,
       24667.419   , 11987.1682  ,  9625.92    , 24915.22085 ,
        1391.5287  ,  4151.0287  ,  3558.62025 ,  8538.28845 ,
        3353.284   ,  8116.26885 ,  7954.517   , 48673.5588  ,
       13981.85035 , 10713.644   , 15817.9857  ,  3554.203   ,
        8733.22925 , 44585.45587 , 39597.4072  ,  1880.07    ,
        4766.022   ,  2866.091   , 21659.9301  , 44400.4064  ,
       36307.7983  ,  2719.27975 , 11015.1747  ,  6272.4772  ,
        4719.52405 , 11830.6072  ,  2473.3341  ,  2331.519   ,
       24915.22085 , 60021.39897 , 11856.4115  , 19673.33573 ,
        3021.80915 ,  8442.667   ,  7726.854   , 12913.9924  ,
        1252.407   , 46130.5265  , 14590.63205 , 25333.

In [15]:
from sklearn.metrics import r2_score
r_score = r2_score(y_test, y_pred)
r_score

0.6935407981017487

In [16]:
prediction = grid.predict([[22,34,0,1,1]])
print('prediction={}'.format(prediction))

prediction=[34779.615]


C:\Users\Admin\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(
